# MIND — an AI that learns from its own mistakes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anusidhu9/ai-that-learns-from-its-mistakes/blob/main/MIND_Colab.ipynb)

A reinforcement-learning agent explores a grid world. It is **rewarded** for reaching the goal and **penalised** for mistakes: hitting walls, stepping on hazards, revisiting cells, and running out of time. This notebook trains three agents on the same maze and checks whether their mistakes actually go down:

| Agent | Idea |
|---|---|
| Random | Baseline that never learns |
| Tabular Q-learning | A table of action values, updated by $Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$ |
| DQN | A neural network approximates $Q$, trained on replayed past experience with a frozen target network |

Runs on the **free Colab CPU**; no GPU needed. Total runtime is about 2–3 minutes.

> This is the starter notebook. Later project stages add checkpoints, multi-seed evaluation, and an interactive dashboard as new sections.

## 1. Setup
Clones the repository (or updates it if already cloned) and makes `src` importable.

In [ ]:
import os, sys

REPO_URL = "https://github.com/anusidhu9/ai-that-learns-from-its-mistakes.git"
REPO_DIR = "ai-that-learns-from-its-mistakes"

if not os.path.exists("src/envs.py"):          # not already inside the repo
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
!git pull -q
!pip install -q pytest

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

## 2. Run the test suite
The quick subset takes about 15 s. Every test should pass; the exact count grows as stages are added.

In [ ]:
!python -m pytest -q -m "not slow"

## 3. The environments

Four presets share one observation format, so every algorithm sees exactly the same information.
`#` wall · `.` free · `H` hazard · `G` goal · `A` agent. Every layout is guaranteed to have a hazard-free path to the goal.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

from src.envs import make_env, PRESETS, WALL
from src.agents import make_agent, EpsilonSchedule
from src.training import train_agent

for name in PRESETS:
    env = make_env(name)
    env.reset(seed=0)
    print(f"{name}: {env.config.height}x{env.config.width}, shortest safe path = "
          f"{env.optimal_length} steps, observation size = {env.obs_size}")
    print(env.render_text(), end="\n\n")

## 4. Shared settings and plotting helpers

In [ ]:
PRESET = "maze"        # try "maze_hazards", "open", "open_hazards"
EPISODES = 400
SEED = 0
WINDOW = 50            # episodes compared at the start vs the end of training

# Chart colours: one fixed colour per agent, checked for colour-blind separation.
COLORS = {"Tabular Q": "#2a78d6", "DQN": "#eb6834", "Random": "#8a8a85"}
INK, MUTED, GRID = "#2b2b29", "#6b6b66", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False,
    "font.size": 10, "lines.linewidth": 2,
})
CELL_CMAP = ListedColormap(["#fcfcfb", "#3d3d3a", "#e34948", "#008300"])  # empty, wall, hazard, goal
HEAT_CMAP = LinearSegmentedColormap.from_list(
    "blues", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])

def rolling(values, window=25):
    values = np.asarray(values, dtype=float)
    out = np.convolve(values, np.ones(window) / window, mode="valid")
    return np.arange(window - 1, len(values)), out

def clean_axes(ax):
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

def draw_grid(ax, env, title=""):
    ax.imshow(env.grid, cmap=CELL_CMAP, vmin=0, vmax=3)
    clean_axes(ax)
    ax.set_title(title)

## 5. Train the three agents
Tabular Q-learning takes a few seconds; DQN takes 1–2 minutes on the free CPU.

In [ ]:
results = {}
schedule = dict(decay_episodes=300)   # epsilon: 1.0 -> 0.05 over 300 episodes

for label, agent_name in [("Random", "random"), ("Tabular Q", "tabular_q"), ("DQN", "dqn")]:
    env = make_env(PRESET)
    kwargs = {} if agent_name == "random" else {"epsilon": EpsilonSchedule(**schedule)}
    agent = make_agent(agent_name, env.obs_size, env.n_actions, seed=SEED, **kwargs)
    start = time.time()
    history = train_agent(env, agent, EPISODES, seed=SEED)
    results[label] = {"env": env, "agent": agent, "history": history}
    print(f"{label:<10} trained {EPISODES} episodes in {time.time() - start:5.1f} s")

## 6. Learning curves

Each line is a 25-episode rolling mean. **Success rate is the primary metric.** A falling mistake count alone can be misleading; for example, an agent that ends episodes early on a hazard also makes fewer mistakes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, res in results.items():
    h = res["history"]
    style = {"color": COLORS[label], "linestyle": "--" if label == "Random" else "-", "label": label}
    x, y = rolling([e["success"] for e in h]);  axes[0].plot(x, y, **style)
    x, y = rolling([e["mistakes"] for e in h]); axes[1].plot(x, y, **style)

axes[0].set_title("Success rate"); axes[0].set_ylim(-0.02, 1.02)
axes[1].set_title("Mistakes per episode")
for ax in axes:
    ax.set_xlabel("Training episode")
    ax.legend(loc="center right")
plt.tight_layout(); plt.show()

## 7. Early vs late training
Mean over the first and last 50 training episodes (exploration is still partly on during training).

In [ ]:
rows = []
for label, res in results.items():
    h = res["history"]
    for phase, part in [(f"first {WINDOW}", h[:WINDOW]), (f"last {WINDOW}", h[-WINDOW:])]:
        rows.append({
            "agent": label, "phase": phase,
            "success rate": np.mean([e["success"] for e in part]),
            "steps": np.mean([e["steps"] for e in part]),
            "total reward": np.mean([e["total_reward"] for e in part]),
            "collisions": np.mean([e["collisions"] for e in part]),
            "hazards": np.mean([e["hazards"] for e in part]),
            "revisits": np.mean([e["revisits"] for e in part]),
            "timeouts": np.mean([e["timeout"] for e in part]),
        })
pd.DataFrame(rows).set_index(["agent", "phase"]).round(2)

## 8. Greedy evaluation (exploration off)

The real test: with ε = 0 the agent must rely only on what it learned. `optimal` is the shortest hazard-free path found by breadth-first search.

In [ ]:
def greedy_path(env, agent):
    obs, _ = env.reset()
    path, done = [env.pos], False
    while not done:
        obs, _, term, trunc, info = env.step(agent.act(obs, greedy=True))
        path.append(env.pos)
        done = term or trunc
    return path, info["episode"]

fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4))
table = []
for ax, (label, res) in zip(axes, results.items()):
    path, ep = greedy_path(res["env"], res["agent"])
    table.append({"agent": label, "success": ep["success"], "steps": ep["steps"],
                  "optimal": ep["optimal_length"], "mistakes": ep["mistakes"]})
    outcome = "reached goal" if ep["success"] else "failed"
    draw_grid(ax, res["env"], f"{label}: {outcome} in {ep['steps']} steps")
    rows_, cols_ = zip(*path)
    ax.plot(cols_, rows_, color=COLORS[label], linewidth=2.5, alpha=0.9)
    ax.scatter([cols_[0]], [rows_[0]], s=80, color=COLORS[label], edgecolor="white", zorder=3)
plt.tight_layout(); plt.show()
pd.DataFrame(table).set_index("agent")

## 9. Where does the agent make mistakes?

Mistakes per cell (collisions, revisits, hazards), first 50 vs last 50 training episodes of tabular Q-learning. Both panels share one colour scale, so a fading hotspot means the agent stopped repeating that mistake.

In [ ]:
def mistake_map(history, env, episodes):
    counts = np.zeros(env.grid.shape)
    for i in episodes:
        for m in history[i]["mistake_log"]:
            counts[m["pos"]] += 1
    return counts / len(episodes)

res = results["Tabular Q"]
h, env = res["history"], res["env"]
early = mistake_map(h, env, range(0, WINDOW))
late = mistake_map(h, env, range(EPISODES - WINDOW, EPISODES))
vmax = max(early.max(), late.max())

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, data, title in [(axes[0], early, f"First {WINDOW} episodes"),
                        (axes[1], late, f"Last {WINDOW} episodes")]:
    ax.imshow(np.where(env.grid == WALL, 1.0, np.nan), cmap=ListedColormap(["#3d3d3a"]))
    im = ax.imshow(np.ma.masked_where(env.grid == WALL, data), cmap=HEAT_CMAP, vmin=0, vmax=vmax)
    clean_axes(ax)
    ax.set_title(f"{title}\n{data.sum():.1f} mistakes per episode")
fig.colorbar(im, ax=axes, shrink=0.8, label="mistakes per episode at this cell")
plt.show()

## 10. What this does and doesn't show

* This is **one seed on one fixed layout**. It shows that learning happens, but it is not yet evidence about reliability or generalisation.
* Mistakes during training are measured with exploration still partly on; section 8 is the fair, exploration-free check.
* Results from the development experiments, to be confirmed by the multi-seed evaluation stage:
  * tabular Q-learning memorises fixed layouts but is close to random on unseen layouts;
  * with non-terminal hazards, tabular Q-learning tends to take a route through one hazard, while DQN finds the safe route;
  * with terminal hazards, DQN can learn to end episodes on a hazard, so mistake counts fall while success stays at zero.

**Next sections (added as the project grows):** persistent logs and checkpoints · multi-seed and unseen-layout evaluation · interactive Gradio dashboard.